In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path.cwd().parent.parent.parent.parent.parent
sys.path.insert(0, str(repo_root))

from SysSimX.components.fmu_comp import FMUComponent

from OMPython import ModelicaSystem
from shutil import move

# Define paths and model names
pkg_file = Path.cwd().parent / 'Modelica/ControlledPendulum/package.mo'
pid_model_name = 'ControlledPendulum.PID_Continuous'
pendulum_model_name = 'ControlledPendulum.Pendulum'
model_names = [pid_model_name, pendulum_model_name]

# Define solver options
euler_solver = "euler"
cvode_solver = "cvode"

# Build Modelica Systems and convert to FMUs
fmu_paths = {}
for model_name in model_names:
    model = ModelicaSystem(str(pkg_file), model_name, commandLineOptions=f"--fmiFlags=s:{euler_solver}")
    model.buildModel()
    fmu_path = model.convertMo2Fmu(fmuType='cs')
    dest_path = Path.cwd().parent / 'FMUs' / f"{model_name.split('.')[-1]}.fmu"
    move(fmu_path, dest_path)
    fmu_paths[model_name] = dest_path


Notification: Automatically loaded package Complex 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package ModelicaServices 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package Modelica 4.0.0 due to usage.



Notification: Automatically loaded package Complex 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package ModelicaServices 4.0.0 due to uses annotation from Modelica.
Notification: Automatically loaded package Modelica 4.0.0 due to usage.




**Test Event Indicator**

In [2]:
pendulum = FMUComponent(name='Pendulum', fmu_path=fmu_paths['ControlledPendulum.Pendulum'])
pendulum.set_parameters(q0=0.3, omega0=0.0)
pendulum.initialize(0.0)

# Event indicator for wall hit at q = 0
def pendulum_wall_hit(comp: FMUComponent) -> float:
    q = comp.get_outputs()['q']
    return q
pendulum.add_event_indicator(name='wall_hit',
                             func=pendulum_wall_hit,
                             direction=-1)

t = 0.0
dt = 0.0001
t_end = 0.2

g_before = pendulum.evaluate_event_indicators()
while t < t_end:
    pendulum.do_step(t, dt)
    t += dt
g_after = pendulum.evaluate_event_indicators()

print(f"Event indicator before step: {g_before['wall_hit']}")
print(f"Event indicator after step: {g_after['wall_hit']}")

pendulum.reset()
pendulum.evaluate_event_indicators()

Event indicator before step: 0.3
Event indicator after step: 0.22568672522568836


FMICallException: fmi2GetReal failed with status 3 (error).

In [ ]:


# Rollback functions for state snapshot and restore
def snapshot_state(comp: FMUComponent):
    state = comp._instance.getFMUState()
    return state
pendulum.snapshot_state = snapshot_state.__get__(pendulum)

def restore_state(comp: FMUComponent, state):
    comp._instance.setFMUState(state)
pendulum.restore_state = restore_state.__get__(pendulum)

# Event handling function
def handle_event(comp: FMUComponent, indicator_name: str):
    q = comp.get_outputs()['q']
    omega = comp.get_outputs()['omega']
    t = comp.t
    if indicator_name == 'wall_hit':
        comp.reset()
        comp.set_parameters(q0=0.0, omega0=-0.8 * omega)
        comp.initialize(t)
        comp.do_step(t, 0.0)
pendulum.handle_event = handle_event.__get__(pendulum)

In [25]:
t = 0.0
dt = 0.001
t_end = 1

while t < t_end:
    # 1) Save snapshot before step
    snap_before = pendulum.snapshot_state()
    
    # 2) Evaluate event indicators before step
    prev = pendulum.evaluate_event_indicators()

    # 3) Do step
    pendulum.do_step(t, dt)

    # 4) Evaluate event indicators after step
    curr = pendulum.evaluate_event_indicators()

    # 5) Detect event crossings
    events = pendulum.detect_event_crossing(prev, curr)

    # 6) Handle events if any
    if events:
        print(f"Event detected at time {t+dt}: {events}")
        # a) Restore snapshot
        pendulum.restore_state(snap_before, t)

        # b) Locate event time (simple bisection)
        t_event, g_event, converged = pendulum.locate_event_bisection(event_name="wall_hit", t_left=t, t_right=t+dt)

        # c) Hanlde event
        pendulum.handle_event(event_name="wall_hit")
        t = t_event
    
    else:
        t += dt

Event detected at time 0.43700000000000033: ['wall_hit']


TypeError: restore_state() takes 2 positional arguments but 3 were given